# Stemming und Lemmatisierung

**Leitfrage:** Findet unsere Wortlistenregel mehr passende Bewertungen, wenn wir unterschiedliche Wortformen zusammenfassen?

**Arbeitsauftrag (10–15 Minuten)**

Führe die Startzellen in Abschnitt 1 aus. Ergänze anschließend zwei Funktionen:

1. [Abschnitt 2.1 · `normalize_tokens()`](#normalize-tokens): Verarbeite die Wörter im gewählten Modus.
2. [Abschnitt 2.2 · `score_review()`](#score-review): Wende denselben Modus auf Review und Wortlisten an und berechne den Score.

[Vergleiche in Abschnitt 3](#compare-results) die Accuracy der vier Varianten auf denselben 100 Reviews. Erkläre anhand einer Review, welcher Worttreffer ihren Score verändert.

## 1. Wortformen und Normalisierung

Führe die Startzellen aus. Wir verwenden dieselben Reviews, denselben Tokenizer und dieselben Wortlisten wie in Notebook 02.

**Stemming** bildet Wortformen auf eine gemeinsame Vergleichsform ab. Ein regelbasierter Stemmer prüft typische Endungen und Bedingungen am verbleibenden Wortteil. Dann entfernt oder ersetzt er die Endung. In der Funktion unten ergeben `loved` und `love` beide den Stamm `lov`. Ein Stamm kann eine Form sein, die so nicht im Wörterbuch steht.

**Lemmatisierung** sucht die Wörterbuchgrundform, das Lemma. Ein Wörterbuchverfahren kann mit der Wortart beginnen, unregelmäßige Formen nachschlagen und bei anderen Formen Beugungsregeln ausprobieren. Ein Beispiel für eine Ausnahme ist `went` mit dem Lemma `go`. Die kleine Tabelle unten bildet den Nachschlageschritt ab. In Abschnitt 4 vergleichen wir die Ergebnisse an einem Review-Satz mit NLTK.

[Porters Stemming-Algorithmus](https://tartarus.org/martin/PorterStemmer/def.txt) · [Morphologische Verarbeitung in WordNet](https://wordnet.princeton.edu/documentation/morphy7wn)

In [ ]:
from pathlib import Path
import re

train_dir = Path('data/imdb/train')


def find_review_paths(path: Path) -> list[Path]:
    paths = []
    for review_path in path.rglob('*.txt'):
        paths.append(review_path)
    return sorted(paths)


def read_review(path: Path) -> str:
    with path.open('r', encoding='utf-8') as file:
        return file.read()


def tokenize_review(text: str) -> list[str]:
    text = text.replace('<br />', ' ')
    return re.findall(r'[A-Za-z]+', text)


review_paths = find_review_paths(train_dir)
if not review_paths:
    raise FileNotFoundError(f'No review files found under {train_dir.resolve()}.')

In [ ]:
positive_words = {
    'amazing', 'beautiful', 'best', 'brilliant', 'charming', 'enjoy', 'enjoyable', 'excellent',
    'fantastic', 'funny', 'good', 'great', 'impressive', 'like', 'love', 'moving', 'perfect',
    'recommend', 'superb', 'wonderful',
}
negative_words = {
    'annoying', 'awful', 'bad', 'bland', 'boring', 'cheap', 'disappointing', 'dull', 'hate',
    'horrible', 'mediocre', 'pointless', 'poor', 'ridiculous', 'silly', 'stupid', 'terrible',
    'ugly', 'unwatchable', 'worst',
}

In [ ]:
def stem_word(word: str) -> str:
    if len(word) > 5 and word.endswith('ing'):
        word = word[:-3]
    elif len(word) > 4 and word.endswith('ed'):
        word = word[:-2]
    elif len(word) > 4 and word.endswith('s') and not word.endswith('ss'):
        word = word[:-1]
    if len(word) > 3 and word.endswith('e'):
        word = word[:-1]
    return word

In [ ]:
lemma_groups = {
    'act': ('acted', 'acting', 'acts'),
    'actor': ('actors',),
    'be': ('am', 'are', 'is', 'was', 'were', 'been', 'being'),
    'enjoy': ('enjoyed', 'enjoying', 'enjoys'),
    'film': ('films',),
    'go': ('went', 'gone', 'goes', 'going'),
    'hate': ('hated', 'hating', 'hates'),
    'like': ('liked', 'liking', 'likes'),
    'love': ('loved', 'loving', 'loves'),
    'movie': ('movies',),
    'recommend': ('recommended', 'recommending', 'recommends'),
    'see': ('saw', 'seen', 'sees', 'seeing'),
    'watch': ('watched', 'watching', 'watches'),
}
lemma_by_form = {}
for lemma, forms in lemma_groups.items():
    for form in forms:
        lemma_by_form[form] = lemma


def lemmatize_word(word: str) -> str:
    if word in lemma_by_form:
        return lemma_by_form[word]
    return word

In [ ]:
for word in ['love', 'loved', 'enjoyed', 'hated', 'went', 'rate', 'rat']:
    print(f'{word:<10} stem={stem_word(word):<9} lemma={lemmatize_word(word)}')

## 2. Normalisierung auf die Wortlistenregel anwenden

<a id="normalize-tokens"></a>

### 2.1 Normalisiere die Tokens

Implementiere `normalize_tokens(tokens, mode)`. Die Eingabe ist eine Liste von Review-Tokens oder eine Menge von Listeneinträgen. Gib immer eine Liste zurück. Bei Review-Tokens bleiben Reihenfolge und Wiederholungen erhalten. Verwende die vorgegebenen Funktionen `stem_word()` und `lemmatize_word()`.

| Modus | Verarbeitung |
| --- | --- |
| `exact` | Tokenliste wie in Notebook 02 |
| `lowercase` | Alle Tokens kleinschreiben |
| `stemming` | Kleinschreiben und jedes Token mit `stem_word()` kürzen |
| `lemmatization` | Kleinschreiben und für jedes Token `lemmatize_word()` aufrufen |

Behandle einen unbekannten Modus mit `ValueError`. Als Kontrolle ergeben `['Loved', 'enjoyed']` im Modus `lemmatization` die Liste `['love', 'enjoy']`.

In [ ]:
def normalize_tokens(tokens: list[str] | set[str], mode: str) -> list[str]:
    raise NotImplementedError('Implement normalize_tokens')

<a id="score-review"></a>

### 2.2 Berechne den Score

Implementiere `score_review(text, mode)`. Tokenisiere den Review-Text. Normalisiere die Tokens und **beide Wortlisten mit demselben Modus**. Übergib die Wortmengen direkt an `normalize_tokens()`. Wandle die Ergebnisse für die Mitgliedschaftsprüfung mit `set()` wieder in Mengen um.

Rufe mit den drei normalisierten Sammlungen die vorgegebene Funktion `score_tokens()` auf und gib ihren Score zurück. Pro Vorkommen zählt ein Treffer in der Positivliste als +1 und einer in der Negativliste als −1.

In [ ]:
def score_tokens(tokens: list[str], positive: set[str], negative: set[str]) -> int:
    score = 0
    for token in tokens:
        if token in positive:
            score += 1
        elif token in negative:
            score -= 1
    return score


def score_review(text: str, mode: str) -> int:
    raise NotImplementedError('Implement score_review')

<a id="compare-results"></a>

## 3. Vergleiche die Varianten

Die **Accuracy** ist der Anteil richtiger Zuordnungen an allen Reviews. Eine Review mit Score 0 bleibt unentschieden und zählt im Nenner mit. Vergleiche zuerst den exakten Abgleich mit Kleinschreibung und danach Kleinschreibung mit den beiden weiteren Varianten.

In [ ]:
def predict_sentiment(score: int) -> str:
    if score > 0:
        return 'pos'
    if score < 0:
        return 'neg'
    return 'undecided'


def evaluate(paths: list[Path], mode: str) -> tuple[int, int, int]:
    correct = incorrect = undecided = 0
    for path in paths:
        prediction = predict_sentiment(score_review(read_review(path), mode))
        if prediction == path.parent.name:
            correct += 1
        elif prediction == 'undecided':
            undecided += 1
        else:
            incorrect += 1
    return correct, incorrect, undecided


modes = ['exact', 'lowercase', 'stemming', 'lemmatization']
print('Mode             Correct  Incorrect  Undecided  Accuracy')
for mode in modes:
    correct, incorrect, undecided = evaluate(review_paths, mode)
    accuracy = correct / len(review_paths)
    print(f'{mode:<16} {correct:>7} {incorrect:>10} {undecided:>10} {accuracy:>9.0%}')

### Wie ändern sich die Tokens?

Das Beispiel zeigt den ersten Satz der Review über *Longstreet*. Vergleiche die drei Ausgaben. In welcher Form erscheint `enjoyed` beim Abgleich mit der Positivliste? Schau auch auf `followed` und `steps`. Die kleine Lemmatisierungstabelle aus Abschnitt 1 enthält nur ausgewählte Wortformen.

In [ ]:
sample_path = train_dir / 'pos' / '10589_10.txt'
sample_text = read_review(sample_path).partition('.')[0] + '.'
sample_tokens = tokenize_review(sample_text)
sample_modes = [
    ('Exact', 'exact'),
    ('Stemming', 'stemming'),
    ('Lemmatization', 'lemmatization'),
]

for label, mode in sample_modes:
    print(f'{label}:')
    print(' · '.join(normalize_tokens(sample_tokens, mode)))


### Welche Wörter ändern eine Entscheidung?

Öffne eine der folgenden Reviews. Suche eine Wortform, deren Verarbeitung den Score verändert. Erkläre mit der Wortliste, warum sich die Vorhersage ändert oder unentschieden bleibt.

In [ ]:
example_paths = [
    train_dir / 'pos' / '12275_9.txt',
    train_dir / 'pos' / '10589_10.txt',
    train_dir / 'neg' / '11886_1.txt',
]

for path in example_paths:
    text = read_review(path)
    print(path.relative_to(train_dir))
    for mode in modes:
        score = score_review(text, mode)
        print(f'  {mode:<16} {score:>2}  {predict_sentiment(score)}')

## 4. Vertiefung mit NLTK

Vergleiche die einfachen Funktionen aus Abschnitt 1 mit dem Porter-Stemmer und dem WordNet-Lemmatizer am ersten Satz der Longstreet-Review. WordNet nutzt dabei die Wortart, die NLTK aus dem Satzkontext bestimmt.

Installiere bei Bedarf NLTK im aktuellen Kernel. Die nächste Zelle lädt fehlende Sprachdaten herunter. Betrachte in der Tabelle besonders `steps` und `successful`: Welche Verfahren verändern die beiden Wörter?

In [ ]:
%pip install nltk==3.9.4

In [ ]:
import nltk
from nltk import pos_tag
from nltk.stem import PorterStemmer, WordNetLemmatizer


resources = [
    ('wordnet', 'corpora/wordnet.zip'),
    ('averaged_perceptron_tagger_eng', 'taggers/averaged_perceptron_tagger_eng/'),
]
for package, resource in resources:
    try:
        nltk.data.find(resource)
    except LookupError:
        nltk.download(package, quiet=True, raise_on_error=True)

porter = PorterStemmer()
wordnet_lemmatizer = WordNetLemmatizer()

In [ ]:
wordnet_tags = {'J': 'a', 'N': 'n', 'R': 'r', 'V': 'v'}

print(sample_text)
print('Token          Simple stem    Porter stem    Simple lemma   WordNet lemma')
for token, tag in pos_tag(sample_tokens):
    word = token.lower()
    pos = wordnet_tags.get(tag[0])
    lemma = wordnet_lemmatizer.lemmatize(word, pos=pos) if pos else word
    print(f'{token:<14} {stem_word(word):<14} {porter.stem(word):<14} '
          f'{lemmatize_word(word):<14} {lemma}')